# 🧠 Transfer Learning com Ollama no Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yanchagas04/personal-assistant/blob/feat/transfer_learning/notebooks/transfer_learning_collab.ipynb)

Este notebook está preparado especificamente para execução no **Google Colab** (com ou sem aceleração de GPU).
Ele implementa a técnica de **Transfer Learning / Customização de Modelos** com a biblioteca **Ollama** em Python, transferindo a identidade, tom informal e vocabulário de **Yan Chagas** para a família de modelos **LLaMA 3**.

---

### 📌 Etapas do Notebook:
1. **Configuração do Ambiente Colab:** Instalação do Ollama via script oficial Linux e da biblioteca Python `ollama`.
2. **Inicialização do Servidor Ollama:** Execução do daemon `ollama serve` em background no Colab.
3. **Download do Modelo Base:** Pull do modelo LLaMA (ex: `llama3.2` ou `llama3.2:1b` para agilidade).
4. **Carregamento dos Dados Processados:** Leitura das 25 conversas do dataset do clone (`chat_dataset_sample25.jsonl`).
5. **Geração do Modelfile:** Definição do System Prompt, hiperparâmetros e injeção dos diálogos de *few-shot learning*.
6. **Criação do Modelo `yan-clone`:** Registro via API Python com `ollama.create()`.
7. **Testes de Inferência e Avaliação:** Comparação de respostas e validação de autenticidade.
8. **Chat Interativo:** Conversação dinâmica em tempo real com o clone.

## 1. Instalação e Inicialização do Ollama no Colab

In [ ]:
# 1. Instala o Ollama no ambiente Linux do Colab
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Instala os pacotes Python necessários
!pip install -q ollama python-dotenv

### Inicializando o Servidor Ollama em Segundo Plano

In [ ]:
import subprocess
import time
import urllib.request

# Inicia o serviço do Ollama em background
ollama_process = subprocess.Popen(["ollama", "serve"])
time.sleep(4)

# Valida conexão com a API local
try:
    with urllib.request.urlopen("http://localhost:11434/") as response:
        print("✅ Servidor Ollama ativo e respondendo:", response.read().decode().strip())
except Exception as e:
    print("⚠️ Aguardando inicialização do servidor...", e)

## 2. Download do Modelo Base no Colab

Vamos baixar o modelo base **`llama3.2`** (ou `llama3.2:1b` se desejar um download ainda mais rápido).

In [ ]:
BASE_MODEL = "llama3.2"

print(f"Baixando modelo base '{BASE_MODEL}' no Ollama...")
!ollama pull {BASE_MODEL}

import ollama
client = ollama.Client()
models = client.list()
print("\nModelos disponíveis:", [m.model for m in models.models])

## 3. Obtenção do Dataset Processado

Clonamos o repositório ou carregamos o arquivo `chat_dataset_sample25.jsonl`.

In [ ]:
import os
from pathlib import Path
import json

# Clona o repositório se ainda não existir no ambiente do Colab
if not Path("personal-assistant").exists() and not Path("data").exists():
    !git clone -b feat/transfer_learning https://github.com/yanchagas04/personal-assistant.git
    dataset_path = Path("personal-assistant/data/processed/chat_dataset_sample25.jsonl")
elif Path("personal-assistant").exists():
    dataset_path = Path("personal-assistant/data/processed/chat_dataset_sample25.jsonl")
else:
    dataset_path = Path("data/processed/chat_dataset_sample25.jsonl")

print(f"Caminho do dataset: {dataset_path}")

with open(dataset_path, "r", encoding="utf-8") as f:
    dataset_dialogues = [json.loads(line) for line in f]

print(f"✅ Total de {len(dataset_dialogues)} conversas carregadas com sucesso!")

## 4. Construção do Modelfile para Transfer Learning

Configuramos o System Prompt com as diretrizes do clone e inserimos os turnos de conversa autêntica (*few-shot in-context learning*).

In [ ]:
system_prompt = (
    "Você é o clone digital autêntico de Yan Chagas. "
    "Responda sempre direto ao ponto, de forma descontraída, autêntica e informal, "
    "exatamente como nas conversas com seus amigos. "
    "Use linguagem natural brasileira com gírias espontâneas (ex: 'zorra', 'massa', 'tranquilo', 'tô na correria'), "
    "evite explicações prolixas e nunca aja como um assistente corporativo."
)

modelfile_lines = [
    f"FROM {BASE_MODEL}",
    "PARAMETER temperature 0.7",
    "PARAMETER top_p 0.9",
    'PARAMETER stop "<|eot_id|>"',
    'PARAMETER stop "<|end_of_text|>"',
    "",
    f'SYSTEM """{system_prompt}"""',
    "",
    "# --- Exemplos de Transfer Learning (Few-Shot) ---"
]

for dialogue in dataset_dialogues[:15]:
    for msg in dialogue.get("messages", []):
        role = msg.get("role")
        if role in ["user", "assistant"]:
            sanitized = msg["content"].replace('"', '\\"')
            modelfile_lines.append(f'MESSAGE {role} "{sanitized}"')

modelfile_content = "\n".join(modelfile_lines)

# Salvar Modelfile no Colab
modelfile_path = Path("Modelfile.yan_clone")
with open(modelfile_path, "w", encoding="utf-8") as f:
    f.write(modelfile_content)

print(f"✅ Modelfile gerado em: {modelfile_path} ({len(modelfile_lines)} linhas)")

## 5. Criação do Modelo Personalizado no Ollama

In [ ]:
CUSTOM_MODEL = "yan-clone"

print(f"Compilando modelo '{CUSTOM_MODEL}' via API do Ollama...")
progress = ollama.create(model=CUSTOM_MODEL, modelfile=modelfile_content, stream=True)
for step in progress:
    status = step.get("status", "")
    if status:
        print("Status:", status)

print(f"\n🎉 Modelo '{CUSTOM_MODEL}' criado com sucesso no Colab!")

## 6. Testes de Inferência e Avaliação do Clone

In [ ]:
perguntas = [
    "Bora almoçar onde hoje?",
    "Conseguiu ver aquele erro no banco?",
    "Vai ter aula hoje na faculdade?",
    "Bora jogar um CS mais tarde?",
    "E aí, vale a pena comprar esse notebook por 5 mil?"
]

for p in perguntas:
    print(f"💬 Usuário: {p}")
    resp = ollama.chat(model=CUSTOM_MODEL, messages=[{"role": "user", "content": p}])
    print(f"🤖 {CUSTOM_MODEL}: {resp['message']['content']}")
    print("-" * 60)

## 7. Chat Interativo no Colab

In [ ]:
def chat(mensagem: str, historico: list = None):
    if historico is None:
        historico = []
    historico.append({"role": "user", "content": mensagem})
    resposta = ollama.chat(model=CUSTOM_MODEL, messages=historico)
    resposta_texto = resposta["message"]["content"]
    historico.append({"role": "assistant", "content": resposta_texto})
    return resposta_texto, historico

# Exemplo de conversa contínua de múltiplos turnos:
msg1, hist = chat("E aí mano, tudo certo?")
print(f"User: E aí mano, tudo certo?\nClone: {msg1}\n")

msg2, hist = chat("Bora marcar de sair amanhã?", hist)
print(f"User: Bora marcar de sair amanhã?\nClone: {msg2}")